In [ ]:
import torch

torch.manual_seed(42)

def grpo_loss(
    logp: torch.Tensor,        # [K]
    logp_ref: torch.Tensor,    # [K]
    reward: torch.Tensor,      # [K]
    beta: float = 0.1,
    eps: float = 1e-6,
):
    reward_mean = reward.mean()
    reward_std = reward.std(unbiased=False)
    advantage = (reward - reward_mean) / (reward_std + eps)

    pg_loss = -(advantage.detach() * logp).mean()
    kl_loss = (logp - logp_ref).mean()

    loss = pg_loss + beta * kl_loss

    return loss, pg_loss, kl_loss, advantage


# ========== 构造一个 fake rollout ==========
K = 4  # number of responses

# 假设这是 policy 输出的 log-prob（需要梯度）
logp = torch.randn(K, requires_grad=True)

# reference policy（通常 frozen）
logp_ref = torch.randn(K)

# reward（来自 reward model / rule / judge）
reward = torch.tensor([0.9, 0.8, 0.2, 0.1])

# ========== 计算 GRPO loss ==========
loss, pg_loss, kl_loss, advantage = grpo_loss(
    logp=logp,
    logp_ref=logp_ref,
    reward=reward,
    beta=0.1,
)

print("logp:      ", logp.detach().numpy())
print("logp_ref:  ", logp_ref.numpy())
print("reward:    ", reward.numpy())
print("advantage: ", advantage.detach().numpy())

print("pg_loss:", pg_loss.item())
print("kl_loss:", kl_loss.item())
print("total loss:", loss.item())

# ========== 反向传播 ==========
loss.backward()

print("\nlogp.grad (policy gradient):")
print(logp.grad)


logp:       [0.33669037 0.1288094  0.23446237 0.23033303]
logp_ref:   [-1.1228564  -0.18632829  2.2082014  -0.63799703]
reward:     [0.9 0.8 0.2 0.1]
advantage:  [ 1.1313676   0.84852576 -0.84852576 -1.1313677 ]

pg_loss: -0.007669992744922638
kl_loss: 0.16731886565685272
total loss: 0.009061893448233604

logp.grad (policy gradient):
tensor([-0.2578, -0.1871,  0.2371,  0.3078])


In [ ]:
import torch

class GRPO:
    
    def __init__(self,eps,clip,beta):
        self.eps =eps
        self.clip = clip
        self.beta = beta
    
    def mask_mean(self,loss,mask,dim=-1):
        return (loss*mask).sum(dim=dim)/mask.sum(dim=dim) # 点乘 和 tensor 按维度求和

    
    def group_advantages(self,rewards):
        mean=torch.mean(rewards)
        std = torch.std(rewards,unbiased=False)# unbiased=False， 真实这组数的尺度，而不是总体估计
        advantages = (rewards-mean)/(std+self.eps)
        return advantages.detach()# 梯度截断
    
    def grpo_loss(self,old_logps, new_logps,ref_logps,advantages):
        ref_logr = new_logps - ref_logps # logr
        kl_score = (torch.exp(ref_logr)-ref_logr-1)*self.beta #  KL penalty: e^logr-logr-1
        ratio = torch.exp(new_logps - old_logps) # r   exp 的作用: 把“对数概率差”还原成“概率的比例”
        surr1 = ratio*advantages # r*A
        surr2 = torch.clamp(ratio,1-self.clip,1+self.clip)*advantages # clip(r)*A
        loss = - torch.mean(torch.min(surr1,surr2)-kl_score) # 最大化,
        return loss # loss
    

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

class DPO:
    def __init__(self,beta):
        self.beta = beta
        
    def dpo_loss(self,policy_chosen_logps, policy_reject_logps, ref_chosen_logps,ref_reject_logps):
        chosen_r = policy_chosen_logps - ref_chosen_logps
        reject_r = policy_reject_logps - ref_reject_logps
        loss = -F.logsigmoid(self.beta*(chosen_r-reject_r))
        return loss.mean()


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

class PPO:
    def __init__(self,clip,gamma=0.99,lam=0.95):
        self.clip = clip
        self.gamma = gamma
        self.lam = lam
    
    def mask_mean(self,loss,mask,dim=-1):
        return (loss*mask).sum(dim=dim)/mask.sum(dim=dim)
    
    def advantage_estimate(self,rewards,values):
        seq_len = values.shape[1]
        advantages = torch.zeros_like(rewards)
        gae=0
        for i in range(seq_len-1,-1,-1):
            next_value = values[:,i+1] if i<seq_len-1 else 0.0
            delta = rewards[:,i]+self.gamma*next_value-values[:,i] # TD error
            gae = delta+self.lam*self.gamma*gae # At​=δt​+γλ*At+1​
            advantages[:,i]=gae # 给 batch 中所有 sequence 的第 i 个 token 赋 advantage
        returns =advantages+values # 
        return advantages,returns
    
    def policy_loss(self,new_probs,old_probs,advantages,act_mask):
        ratio = torch.exp(new_probs-old_probs)
        surr1=ratio*advantages
        surr2=torch.clamp(ratio,1-self.clip,1+self.clip)*advantages
        loss=-torch.min(surr1,surr2)
        return self.mask_mean(loss,act_mask) # act_mask 是 action mask（动作mask）
    
    def value_loss(self,new_values,returns,act_mask):
        loss = (new_values-returns)**2
        return self.mask_mean(loss,act_mask)

In [ ]:
import torch

class GRPO:
    
    def __init__(self,eps,clip,beta):
        self.eps =eps
        self.clip = clip
        self.beta = beta
    
    def mask_mean(self,loss,mask,dim=-1):
        return (loss*mask).sum(dim=dim)/mask.sum(dim=dim) # 点乘 和 tensor 按维度求和

    
    def group_advantages(self,rewards):
        mean=torch.mean(rewards)
        std = torch.std(rewards,unbiased=False)# unbiased=False， 真实这组数的尺度，而不是总体估计
        advantages = (rewards-mean)/(std+self.eps)
        return advantages.detach()# 梯度截断
    
    def grpo_loss(self,old_logps, new_logps,ref_logps,advantages,act_mask):
        ref_logr = new_logps - ref_logps # logr
        kl_score = (torch.exp(ref_logr)-ref_logr-1)*self.beta #  KL penalty: e^logr-logr-1
        ratio = torch.exp(new_logps - old_logps) # r   exp 的作用: 把“对数概率差”还原成“概率的比例”
        surr1 = ratio*advantages # r*A
        surr2 = torch.clamp(ratio,1-self.clip,1+self.clip)*advantages # clip(r)*A
        loss = - (torch.min(surr1,surr2)-kl_score) # 最大化回报
        return self.mask_mean(loss,act_mask) # loss
    

In [ ]:
import torch

class GRPO:

    def __init__(self,eps,clip,beta):
        self.eps=eps
        self.clip=clip
        self.beta=beta
    
    def mask_mean(self,loss,mask,dim=-1):
        return loss*mask.sum(dim=dim)/mask.sum(dim=dim)
    
    def advantage(self,rewards):
        mean=torch.mean(rewards)
        std = torch.std(rewards,unbiaed = False)
        advantages = rewards-mean/(std+self.eps)
        return advantages.detach()
    
    def grpo_loss(self,old_logps,new_logps,ref_logps,advantages,act_mask):
        ratio = torch.exp(new_logps-old_logps)
        surr1=ratio*advantages
        surr2= torch.clamp(ratio,1-self.clip,1+self.clip)*advantages
        

        # KL
        ref_logr = torch.exp(new_logps-ref_logps)


In [1]:
import torch

class GRPO:
    
    def __init__(self, eps, clip, beta):
        self.eps = eps
        self.clip = clip
        self.beta = beta
    
    def mask_mean(self, loss, mask, dim=-1):
        """
        Compute masked mean along given dimension

        loss shape: (batch, seq_len)
        mask shape: (batch, seq_len)

        element-wise multiply keeps only valid positions
        """
        return (loss * mask).sum(dim=dim) / mask.sum(dim=dim)

    
    def group_advantages(self, rewards):
        """
        Normalize rewards to compute advantages

        rewards shape: (batch,)
        output shape: (batch,)
        """

        mean = torch.mean(rewards)
        std = torch.std(rewards, unbiased=False)

        advantages = (rewards - mean) / (std + self.eps)

        # detach to stop gradient propagation
        return advantages.detach()

    
    def grpo_loss(self, old_logps, new_logps, ref_logps, advantages, act_mask):
        """
        old_logps shape: (batch, seq_len)
        new_logps shape: (batch, seq_len)
        ref_logps shape: (batch, seq_len)

        advantages shape: (batch, 1) or (batch,)
        act_mask shape: (batch, seq_len)

        PyTorch broadcasting will automatically expand advantages
        """

        # KL penalty term
        ref_logr = new_logps - ref_logps
        kl_score = (torch.exp(ref_logr) - ref_logr - 1) * self.beta

        # policy ratio
        ratio = torch.exp(new_logps - old_logps)

        # Broadcasting happens here:
        #
        # ratio shape        : (batch, seq_len)
        # advantages shape   : (batch, 1)
        #
        # advantages automatically broadcast to:
        #
        #                    : (batch, seq_len)
        #
        surr1 = ratio * advantages

        surr2 = torch.clamp(
            ratio,
            1 - self.clip,
            1 + self.clip
        ) * advantages

        loss = -(torch.min(surr1, surr2) - kl_score)

        return self.mask_mean(loss, act_mask)


# ==========================
# Test Code
# ==========================

def test_broadcast_and_grpo():

    torch.manual_seed(0)

    batch_size = 3
    seq_len = 5

    grpo = GRPO(
        eps=1e-8,
        clip=0.2,
        beta=0.01
    )

    # simulate log probabilities
    old_logps = torch.randn(batch_size, seq_len)

    # requires_grad=True to verify gradient flow
    new_logps = torch.randn(
        batch_size,
        seq_len,
        requires_grad=True
    )

    ref_logps = torch.randn(batch_size, seq_len)

    # simulate rewards
    rewards = torch.randn(batch_size)

    print("\n=== rewards shape ===")
    print(rewards.shape)   # (batch,)

    # compute advantages
    advantages = grpo.group_advantages(rewards)

    print("\n=== advantages BEFORE reshape ===")
    print(advantages.shape)  # (batch,)

    # reshape to (batch, 1)
    advantages = advantages.unsqueeze(1)

    print("\n=== advantages AFTER reshape ===")
    print(advantages.shape)  # (batch, 1)

    print("\n=== new_logps shape ===")
    print(new_logps.shape)   # (batch, seq_len)

    # Demonstrate broadcasting explicitly
    broadcast_result = new_logps * advantages

    print("\n=== broadcast_result shape ===")
    print(broadcast_result.shape)
    print("Explanation:")
    print("(batch, seq_len) * (batch, 1)")
    print("-> (batch, seq_len) via broadcasting")

    # simulate mask
    act_mask = torch.randint(
        0, 2,
        (batch_size, seq_len)
    ).float()

    print("\n=== mask ===")
    print(act_mask)

    # compute loss
    loss = grpo.grpo_loss(
        old_logps,
        new_logps,
        ref_logps,
        advantages,
        act_mask
    )

    print("\n=== loss per sample ===")
    print(loss)

    total_loss = loss.mean()

    print("\n=== total loss ===")
    print(total_loss.item())

    # backward pass
    total_loss.backward()

    print("\n=== gradient shape ===")
    print(new_logps.grad.shape)

    print("\n=== gradient ===")
    print(new_logps.grad)

    print("\nSUCCESS: Broadcasting and gradient flow verified")


# run test
if __name__ == "__main__":
    test_broadcast_and_grpo()


=== rewards shape ===
torch.Size([3])

=== advantages BEFORE reshape ===
torch.Size([3])

=== advantages AFTER reshape ===
torch.Size([3, 1])

=== new_logps shape ===
torch.Size([3, 5])

=== broadcast_result shape ===
torch.Size([3, 5])
Explanation:
(batch, seq_len) * (batch, 1)
-> (batch, seq_len) via broadcasting

=== mask ===
tensor([[0., 1., 1., 0., 1.],
        [0., 0., 1., 0., 0.],
        [0., 1., 1., 0., 1.]])

=== loss per sample ===
tensor([ 3.9476, -1.5408,  0.6483], grad_fn=<DivBackward0>)

=== total loss ===
1.0183541774749756

=== gradient shape ===
torch.Size([3, 5])

=== gradient ===
tensor([[ 0.0000e+00,  1.2322e-03,  1.1695e+00,  0.0000e+00,  1.1481e-03],
        [ 0.0000e+00,  0.0000e+00, -5.0881e-01,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00, -8.8192e-04,  6.9122e-02,  0.0000e+00,  9.3414e-02]])

SUCCESS: Broadcasting and gradient flow verified


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F


class PPO:
    def __init__(self, clip, gamma=0.99, lam=0.95):
        self.clip = clip
        self.gamma = gamma
        self.lam = lam

    def mask_mean(self, loss, mask, dim=-1):
        """
        Compute masked mean

        loss shape : (batch, seq_len)
        mask shape : (batch, seq_len)

        Only positions where mask=1 contribute
        """
        return (loss * mask).sum(dim=dim) / mask.sum(dim=dim)

    def advantage_estimate(self, rewards, values):
        """
        Compute GAE (Generalized Advantage Estimation)

        rewards shape : (batch, seq_len)
        values shape  : (batch, seq_len)

        returns:
            advantages shape : (batch, seq_len)
            returns shape    : (batch, seq_len)
        """

        seq_len = values.shape[1]

        advantages = torch.zeros_like(rewards)

        gae = 0

        # backward computation (time dimension)
        for i in range(seq_len - 1, -1, -1):

            next_value = values[:, i + 1] if i < seq_len - 1 else 0.0

            # TD error
            delta = rewards[:, i] + self.gamma * next_value - values[:, i]

            # GAE recursion
            gae = delta + self.gamma * self.lam * gae

            advantages[:, i] = gae

        returns = advantages + values

        return advantages, returns

    def policy_loss(self, new_log_probs, old_log_probs, advantages, act_mask):
        """
        new_log_probs shape : (batch, seq_len)
        old_log_probs shape : (batch, seq_len)
        advantages shape    : (batch, seq_len) or (batch, 1)
        act_mask shape      : (batch, seq_len)

        Broadcasting demonstration:
        if advantages shape is (batch, 1),
        PyTorch automatically expands to (batch, seq_len)
        """

        ratio = torch.exp(new_log_probs - old_log_probs)

        print("\n=== Broadcasting demonstration inside policy_loss ===")
        print("ratio shape:", ratio.shape)
        print("advantages shape:", advantages.shape)

        surr1 = ratio * advantages
        surr2 = torch.clamp(
            ratio,
            1 - self.clip,
            1 + self.clip
        ) * advantages

        print("surr1 shape after broadcast:", surr1.shape)

        loss = -torch.min(surr1, surr2)

        return self.mask_mean(loss, act_mask)

    def value_loss(self, new_values, returns, act_mask):
        """
        new_values shape : (batch, seq_len)
        returns shape    : (batch, seq_len)

        MSE loss with mask
        """

        loss = (new_values - returns) ** 2

        return self.mask_mean(loss, act_mask)


# ==========================================
# Test Code
# ==========================================

def test_ppo():

    torch.manual_seed(0)

    batch_size = 2
    seq_len = 4
    hidden_dim = 8

    ppo = PPO(clip=0.2)

    # Fake observations
    obs = torch.randn(batch_size, seq_len, hidden_dim)

    # Simple policy network
    policy_net = nn.Linear(hidden_dim, 1)

    # Simple value network
    value_net = nn.Linear(hidden_dim, 1)

    # Forward pass
    new_log_probs = policy_net(obs).squeeze(-1)
    old_log_probs = new_log_probs.detach() + 0.1 * torch.randn_like(new_log_probs)

    new_values = value_net(obs).squeeze(-1)

    # Fake rewards
    rewards = torch.randn(batch_size, seq_len)

    # Action mask
    act_mask = torch.randint(
        0, 2,
        (batch_size, seq_len)
    ).float()

    print("\n=== Shapes ===")
    print("obs:", obs.shape)
    print("new_log_probs:", new_log_probs.shape)
    print("new_values:", new_values.shape)
    print("rewards:", rewards.shape)
    print("mask:", act_mask.shape)

    # Compute GAE
    advantages, returns = ppo.advantage_estimate(
        rewards,
        new_values.detach()
    )

    print("\n=== GAE Results ===")
    print("advantages shape:", advantages.shape)
    print("returns shape:", returns.shape)

    # =====================================
    # Broadcasting demonstration
    # =====================================

    # Convert advantages to (batch, 1)
    advantages_broadcast = advantages.mean(dim=1, keepdim=True)

    print("\n=== Broadcasting demo ===")
    print("advantages_broadcast shape:", advantages_broadcast.shape)

    broadcast_result = new_log_probs * advantages_broadcast

    print("new_log_probs shape:", new_log_probs.shape)
    print("result shape after broadcast:", broadcast_result.shape)

    print("\nExplanation:")
    print("(batch, seq_len) * (batch, 1)")
    print("-> automatically broadcast to (batch, seq_len)")

    # =====================================
    # Compute losses
    # =====================================

    policy_loss = ppo.policy_loss(
        new_log_probs,
        old_log_probs,
        advantages_broadcast,
        act_mask
    )

    value_loss = ppo.value_loss(
        new_values,
        returns,
        act_mask
    )

    total_loss = (policy_loss + value_loss).mean()

    print("\n=== Loss ===")
    print("policy_loss:", policy_loss)
    print("value_loss:", value_loss)
    print("total_loss:", total_loss.item())

    # =====================================
    # Backward
    # =====================================

    total_loss.backward()

    print("\n=== Gradients ===")
    print("policy_net weight grad shape:",
          policy_net.weight.grad.shape)

    print("value_net weight grad shape:",
          value_net.weight.grad.shape)

    print("\nSUCCESS: PPO test passed")


# run test
if __name__ == "__main__":
    test_ppo()